# Notebook Security and Usage Guardrails
- Do not store secrets, passwords, tokens, or keys in notebook source or outputs.
- Load credentials only from environment variables (`.env`) and keep outputs cleared before commit.
- Canonical production logic lives under `data_pipeline/src`; this notebook is for exploration/reference.


In [ ]:
import os
from pathlib import Path

REQUIRED_ENV_VARS = ["BGG_USERNAME", "BGG_PASSWORD"]
missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise ValueError(f"Missing required environment variable(s): {', '.join(missing)}")

project_root = Path.cwd()
if not (project_root / "data").exists():
    raise ValueError("Expected ./data directory at repository root; run notebook from repo root.")

print("Environment guard check passed.")


In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload

# Imports and Loads

In [ ]:
from bs4 import BeautifulSoup
import os
import requests
from zipfile import ZipFile
from io import BytesIO
from datetime import datetime
import pandas as pd
import logging
import csv
from pathlib import Path
from time import sleep

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from dotenv import load_dotenv, find_dotenv

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("../../logs/get_ranks.log"), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

# Functions

In [ ]:
def get_driver_and_cookies():
    """
    Initialize Selenium WebDriver and authenticate with BoardGameGeek.
    Returns authentication cookies needed for subsequent API requests.

    Returns:
        dict: Authentication cookies for BGG API requests
    """
    logger.info("Initializing web driver and getting cookies")
    LOGIN_USERNAME_FIELD = '//*[@id="inputUsername"]'
    LOGIN_PASSWORD_FIELD = '//*[@id="inputPassword"]'
    LOGIN_BUTTON = '//*[@id="mainbody"]/div/div/gg-login-page/div[1]/div/gg-login-form/form/fieldset/div[3]/button[1]'

    load_dotenv(find_dotenv())
    USERNAME = os.getenv("BGG_USERNAME")
    PASSWORD = os.getenv("BGG_PASSWORD")
    if not USERNAME or not PASSWORD:
        logger.error("Missing BGG_USERNAME or BGG_PASSWORD in environment")
        raise ValueError("Missing BGG_USERNAME or BGG_PASSWORD in environment")

    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    cookies = {}

    driver = webdriver.Chrome(
        service=Service("/usr/lib/chromium-browser/chromedriver"),
        options=chrome_options,
    )
    logger.info("Chrome driver initialized successfully")

    driver.get("https://boardgamegeek.com/login")
    login = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_USERNAME_FIELD))
    )
    password = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_PASSWORD_FIELD))
    )

    login_button = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_BUTTON))
    )

    login.send_keys(USERNAME)
    password.send_keys(PASSWORD)

    login_button.click()
    sleep(1)
    logger.info("Successfully logged in to BoardGameGeek")

    selenium_cookies = driver.get_cookies()
    for cookie in selenium_cookies:
        cookies[cookie["name"]] = cookie["value"]
    logger.info("Successfully retrieved cookies")
    return cookies

# Code

In [ ]:
cookies = get_driver_and_cookies()

In [ ]:
logger.info("Fetching boardgame ranks")
bg_ranks_pg_url = "https://boardgamegeek.com/data_dumps/bg_ranks"
resp = requests.get(bg_ranks_pg_url, cookies=cookies)
soup = BeautifulSoup(resp.content, "html.parser")
bg_ranks_url = soup.find("div", {"id": "maincontent"})("a")[0]["href"]
bg_ranks_zip = requests.get(bg_ranks_url)
queried_at_utc = datetime.now().replace(microsecond=0).isoformat()

In [ ]:
save_file = True
with ZipFile(BytesIO(bg_ranks_zip.content)) as archive:
    with archive.open("boardgames_ranks.csv") as csv_file:
        df_bg_ranks = pd.read_csv(csv_file)
        df_bg_ranks["name"] = df_bg_ranks["name"].str.replace("[“”]", '"', regex=True)
        df_bg_ranks["queried_at_utc"] = queried_at_utc
        logger.info(f"Successfully loaded {len(df_bg_ranks)} boardgames")
        if save_file:
            # Create data directory if it doesn't exist
            data_dir = Path(__file__).parent.parent.parent / "data" / "crawler"
            data_dir.mkdir(parents=True, exist_ok=True)
            
            output_file = data_dir / f"boardgame_ranks_{queried_at_utc[:10].replace('-','')}.csv"
            df_bg_ranks.to_csv(
                output_file,
                index=False,
                sep="|",
                escapechar="\\",
                quoting=csv.QUOTE_NONE,
            )
            logger.info(f"Saved rankings to {output_file}")

In [ ]:
df_bg_ranks.head()